In [4]:
from difflib import SequenceMatcher
import numpy as np
import os
import subprocess
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
PROJECT_ROOT = os.environ["PROJECT_ROOT"]

In [5]:
def align_transcriptions(reference_file, ocr_file):
    """Align reference and OCR transcriptions word by word."""
    
    # Read both files
    with open(reference_file, 'r', encoding='utf-8') as f:
        ref_lines = [line.strip() for line in f.readlines()]
    
    with open(ocr_file, 'r', encoding='utf-8') as f:
        ocr_lines = [line.strip() for line in f.readlines()]
    
    # Concatenate all text
    ref_text = ' '.join(ref_lines)
    ocr_text = ' '.join(ocr_lines)
    
    # Tokenize into words
    ref_words = ref_text.split()
    ocr_words = ocr_text.split()
    
    # Use SequenceMatcher to find alignment
    matcher = SequenceMatcher(None, ref_words, ocr_words)
    
    # Get matching blocks
    alignments = []
    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        alignments.append({
            'tag': tag,  # 'replace', 'delete', 'insert', 'equal'
            'ref_words': ref_words[i1:i2],
            'ocr_words': ocr_words[j1:j2],
            'ref_span': (i1, i2),
            'ocr_span': (j1, j2)
        })
    
    return alignments, ref_words, ocr_words


In [ ]:
alignments, ref_words, ocr_words = align_transcriptions(
    Path(PROJECT_ROOT, 'data/raw/AlbucE.txt'),
    Path(PROJECT_ROOT, 'data/processed/transcription/ocr_kept_20260514_214446/05_garde_001_full.txt'))


In [8]:
alignments

[{'tag': 'replace',
  'ref_words': ['Ayssi'],
  'ocr_words': ['yssi'],
  'ref_span': (0, 1),
  'ocr_span': (0, 1)},
 {'tag': 'equal',
  'ref_words': ['comensan', 'las', 'paraulas', 'de'],
  'ocr_words': ['comensan', 'las', 'paraulas', 'de'],
  'ref_span': (1, 5),
  'ocr_span': (1, 5)},
 {'tag': 'replace',
  'ref_words': ['Albucasim', '.', 'O', ',', 'filhs', ',', 'pus', 'yeu', 'he'],
  'ocr_words': ['de',
   'albucasin',
   'delu',
   'so',
   'conseqͥt',
   'per',
   'aquo',
   'fill',
   'necessari',
   'es'],
  'ref_span': (5, 14),
  'ocr_span': (5, 15)},
 {'tag': 'equal',
  'ref_words': ['a'],
  'ocr_words': ['a'],
  'ref_span': (14, 15),
  'ocr_span': (15, 16)},
 {'tag': 'replace',
  'ref_words': ['vos',
   'complit',
   'aquest',
   'libre',
   ',',
   'le',
   'qual',
   'es',
   'le',
   'derier',
   'de',
   'la',
   'sciencia',
   'de',
   'medicina',
   ',',
   'per',
   'le',
   'compliment',
   'de',
   'lu',
   'so',
   'consequit',
   'la',
   'fi',
   'en',
   'lu',
   '

In [7]:
for i, align in enumerate(alignments[:20]):  # First 20 blocks
    print(f"{i:3d} | {align['tag']:8s} | Ref: {' '.join(align['ref_words'][:5])}... | OCR: {' '.join(align['ocr_words'][:5])}...")

  0 | replace  | Ref: Ayssi... | OCR: yssi...
  1 | equal    | Ref: comensan las paraulas de... | OCR: comensan las paraulas de...
  2 | replace  | Ref: Albucasim . O , filhs... | OCR: de albucasin delu so conseqͥt...
  3 | equal    | Ref: a... | OCR: a...
  4 | replace  | Ref: vos complit aquest libre ,... | OCR: la fe eulu. eper...
  5 | equal    | Ref: las exposicios... | OCR: las exposicios...
  6 | replace  | Ref: de lu , e per... | OCR: delu. eper...
  7 | equal    | Ref: las... | OCR: las...
  8 | replace  | Ref: declaracios de lu , es... | OCR: de claracios delu es ami...
  9 | equal    | Ref: que yeu complesca... | OCR: que yeu complesca...
 10 | replace  | Ref: aquel... | OCR: aquela...
 11 | equal    | Ref: a... | OCR: a...
 12 | replace  | Ref: vos am aquest... | OCR: uos an a q̃st...
 13 | equal    | Ref: tractat... | OCR: tractat...
 14 | replace  | Ref: , le qual... | OCR: lequal...
 15 | equal    | Ref: es partida... | OCR: es partida...
 16 | replace  | Ref: de la... |

In [20]:
from difflib import SequenceMatcher

def align_reference_to_ocr_lines(reference_file, ocr_file, output_file=None):
    """
    Reformat reference text to match OCR line-by-line structure.
    Returns which reference words correspond to each OCR line.
    """
    
    # Read files
    with open(reference_file, 'r', encoding='utf-8') as f:
        ref_text = f.read()
        ref_words = ref_text.split()
    
    with open(ocr_file, 'r', encoding='utf-8') as f:
        ocr_lines = [line.strip() for line in f.readlines()]
    
    # Track position in reference text
    ref_pos = 0
    alignments = []
    
    for ocr_line_num, ocr_line in enumerate(ocr_lines, 1):
        ocr_words = ocr_line.split()
        if not ocr_words:
            continue
        
        # Find best matching span in reference
        best_match = None
        best_score = 0.3  # Minimum similarity threshold
        best_ref_start = None
        best_ref_end = None
        
        # Sliding window search
        for window_size in range(len(ocr_words) - 2, len(ocr_words) + 3):
            if window_size < 1:
                continue
            
            for ref_start in range(ref_pos, min(ref_pos + 50, len(ref_words) - window_size + 1)):
                ref_span = ref_words[ref_start:ref_start + window_size]
                similarity = SequenceMatcher(None, ocr_words, ref_span).ratio()
                
                if similarity > best_score:
                    best_score = similarity
                    best_match = ref_span
                    best_ref_start = ref_start
                    best_ref_end = ref_start + window_size
        
        if best_match:
            alignments.append({
                'ocr_line_num': ocr_line_num,
                'ocr_text': ocr_line,
                'ref_words': best_match,
                'ref_text': ' '.join(best_match),
                'ref_start_idx': best_ref_start,
                'ref_end_idx': best_ref_end,
                'similarity': best_score
            })
            # Move reference position forward
            ref_pos = best_ref_end
        else:
            alignments.append({
                'ocr_line_num': ocr_line_num,
                'ocr_text': ocr_line,
                'ref_words': [],
                'ref_text': '[NO MATCH]',
                'ref_start_idx': None,
                'ref_end_idx': None,
                'similarity': 0.0
            })
    
    # Save reformatted reference
    if output_file:
        with open(output_file, 'w', encoding='utf-8') as f:
            for align in alignments:
                f.write(align['ref_text'] + '\n')
    
    return alignments


In [22]:
# Usage
alignments = align_reference_to_ocr_lines(
    Path(PROJECT_ROOT, 'data/raw/AlbucE.txt'),
    Path(PROJECT_ROOT, 'data/processed/transcription/ocr_kept_20260514_214446/05_garde_001_full.txt'),
    Path(PROJECT_ROOT, 'tests/ocr/05_garde_001_reference_aligned.txt')
)

# View results
for align in alignments[:15]:
    print(f"Line {align['ocr_line_num']:3d} | Score: {align['similarity']:.2f}")
    print(f"  OCR: {align['ocr_text']}")
    print(f"  Ref: {align['ref_text']}")
    print()

Line   1 | Score: 0.89
  OCR: yssi comensan las paraulas de
  Ref: comensan las paraulas de

Line   2 | Score: 0.67
  OCR: de albucasin
  Ref: de

Line   3 | Score: 0.50
  OCR: delu so conseqͥt
  Ref: so

Line   4 | Score: 0.43
  OCR: per aquo fill necessari es a
  Ref: per las declaracios de lu , es a

Line   5 | Score: 0.33
  OCR: la fe eulu. eper
  Ref: de la

Line   6 | Score: 0.00
  OCR: las exposicios
  Ref: [NO MATCH]

Line   7 | Score: 0.00
  OCR: delu. eper las de claracios delu es ami uist
  Ref: [NO MATCH]

Line   8 | Score: 0.00
  OCR: que yeu complesca aquela a uos an a q̃st
  Ref: [NO MATCH]

Line   9 | Score: 0.31
  OCR: tractat lequal es partida dela operacio amn
  Ref: es cyrurgia . Quar la operacio

Line  10 | Score: 0.00
  OCR: ma so es corurgia. quar la operaçio am ma
  Ref: [NO MATCH]

Line  11 | Score: 0.71
  OCR: des prostrada en nostre religio. e en nostie
  Ref: prostrada en nostre regio e en

Line  12 | Score: 0.71
  OCR: temmps de tot priuada entro que fort 

In [16]:
alignments[10]

{'tag': 'replace',
 'ref_words': ['aquel'],
 'ocr_words': ['aquela'],
 'ref_span': (64, 65),
 'ocr_span': (34, 35)}

In [9]:
from difflib import SequenceMatcher
import pandas as pd

def fuzzy_align_lines(reference_file, ocr_file, threshold=0.3):
    """Align OCR lines to reference lines using fuzzy matching."""
    
    with open(reference_file, 'r', encoding='utf-8') as f:
        ref_lines = [line.strip() for line in f.readlines()]
    
    with open(ocr_file, 'r', encoding='utf-8') as f:
        ocr_lines = [line.strip() for line in f.readlines()]
    
    alignments = []
    used_ref_indices = set()
    
    for ocr_idx, ocr_line in enumerate(ocr_lines):
        best_match = None
        best_score = threshold
        best_ref_idx = None
        
        for ref_idx, ref_line in enumerate(ref_lines):
            if ref_idx in used_ref_indices:
                continue
            
            # Calculate similarity
            similarity = SequenceMatcher(None, ocr_line, ref_line).ratio()
            
            if similarity > best_score:
                best_score = similarity
                best_match = ref_line
                best_ref_idx = ref_idx
        
        if best_match is not None:
            alignments.append({
                'ocr_line_num': ocr_idx + 1,
                'ocr_text': ocr_line,
                'ref_line_num': best_ref_idx + 1,
                'ref_text': best_match,
                'similarity': best_score
            })
            used_ref_indices.add(best_ref_idx)
        else:
            alignments.append({
                'ocr_line_num': ocr_idx + 1,
                'ocr_text': ocr_line,
                'ref_line_num': None,
                'ref_text': None,
                'similarity': 0.0
            })
    
    return pd.DataFrame(alignments)



In [10]:
df_alignments = fuzzy_align_lines(
    Path(PROJECT_ROOT, 'data/raw/AlbucE.txt'),
    Path(PROJECT_ROOT, 'data/processed/transcription/ocr_kept_20260514_214446/05_garde_001_full.txt'))

# View results
print(df_alignments[df_alignments['similarity'] > 0.5].head(20))

# Save to CSV
df_alignments.to_csv('line_alignments.csv', index=False)

    ocr_line_num                                      ocr_text  ref_line_num  \
0              1                 yssi comensan las paraulas de             1   
3              4                  per aquo fill necessari es a          1395   
6              7  delu. eper las de claracios delu es ami uist           586   
9             10    ma so es corurgia. quar la operaçio am ma          1645   
11            12   temmps de tot priuada entro que fort leu pe          1660   
13            14      e no romasero delu sino alcimnas petitas           185   
15            16         nuidero las mas. e en deuenc adaquels          2547   
16            17    error. e heyssitacio entro que son clausas           417   
20            21       quest tractat en aquela segon la nia de          2051   
21            22   exposicio. ede declaracio. ede abiemacio. e          1697   
22            23           perque uengua aulas formas dels fer          1698   
24            25    iꝰ mentz de cautheri

In [11]:
def visualize_alignment(ref_text, ocr_text, context=50):
    """Visualize character-level alignment."""
    
    matcher = SequenceMatcher(None, ref_text, ocr_text)
    
    print(f"{'Ref':<6} | {'OCR':<6} | {'Tag':<10} | {'Text'}")
    print("-" * 80)
    
    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        ref_segment = ref_text[i1:i2]
        ocr_segment = ocr_text[j1:j2]
        
        # Truncate for display
        ref_display = ref_segment[:context] + "..." if len(ref_segment) > context else ref_segment
        ocr_display = ocr_segment[:context] + "..." if len(ocr_segment) > context else ocr_segment
        
        print(f"{i1:6d} | {j1:6d} | {tag:<10} | Ref: {ref_display!r}")
        print(f"{'':6s} | {'':6s} | {'':10s} | OCR: {ocr_display!r}")
        print()

# Usage
with open(Path(PROJECT_ROOT, 'data/raw/AlbucE.txt'), 'r') as f:
    ref = f.read()
with open(Path(PROJECT_ROOT, 'data/processed/transcription/ocr_kept_20260514_214446/05_garde_001_full.txt'), 'r') as f:
    ocr = f.read()

visualize_alignment(ref, ocr)

Ref    | OCR    | Tag        | Text
--------------------------------------------------------------------------------
     0 |      0 | delete     | Ref: 'A'
       |        |            | OCR: ''

     1 |      0 | equal      | Ref: 'yssi comensan las paraulas de'
       |        |            | OCR: 'yssi comensan las paraulas de'

    30 |     29 | replace    | Ref: ' A'
       |        |            | OCR: '\nde a'

    32 |     34 | equal      | Ref: 'lbucasi'
       |        |            | OCR: 'lbucasi'

    39 |     41 | replace    | Ref: 'm . \nO ,'
       |        |            | OCR: 'n\ndelu so conseqͥt\nper aquo'

    47 |     68 | equal      | Ref: ' fil'
       |        |            | OCR: ' fil'

    51 |     72 | insert     | Ref: ''
       |        |            | OCR: 'l necessari es a\nla fe eulu. eper\nlas exposicios\nd...'

    51 |    565 | equal      | Ref: 'h'
       |        |            | OCR: 'h'

    52 |    566 | replace    | Ref: 's , pus '
       |        |  

In [23]:
import argparse
import re
from difflib import SequenceMatcher

In [24]:
def tokenize(text: str) -> list[str]:
    """Split text into word tokens (lowercased, punctuation kept attached)."""
    return text.split()

def normalize(text: str) -> str:
    """Light normalisation for matching: lowercase, collapse whitespace."""
    return re.sub(r"\s+", " ", text.lower().strip())

In [27]:
def align_reference_to_lines(trans_lines: list[str], ref_text: str) -> list[str]:
    """
    For each line in trans_lines, find the best matching span in ref_text
    and return that span as the aligned reference line.
 
    Strategy:
      1. Tokenise both the full reference and each transcription line.
      2. Use a sliding window over the reference tokens whose width equals
         the number of tokens in the transcription line (±tolerance).
      3. Pick the window with the highest SequenceMatcher ratio.
      4. Advance a cursor so matches are always forward in the reference.
    """
    ref_tokens = tokenize(ref_text)
    ref_norm   = [normalize(t) for t in ref_tokens]
 
    aligned = []
    cursor  = 0          # we never go backwards — reference is consumed left→right
    tolerance = 10       # tokens we allow window to grow/shrink vs line length
 
    for line_idx, trans_line in enumerate(trans_lines):
        trans_line = trans_line.strip()
        if not trans_line:
            aligned.append("")
            continue
 
        t_tokens = tokenize(trans_line)
        t_norm   = [normalize(t) for t in t_tokens]
        n        = len(t_tokens)
 
        best_ratio = -1.0
        best_start = cursor
        best_end   = min(cursor + n, len(ref_tokens))
 
        # search window: from cursor to cursor + 3×line_length (don't go too far)
        search_end = min(len(ref_tokens), cursor + n * 3 + tolerance)
 
        for start in range(cursor, search_end):
            for width in range(max(1, n - tolerance), n + tolerance + 1):
                end = start + width
                if end > len(ref_tokens):
                    break
                window = ref_norm[start:end]
                ratio  = SequenceMatcher(None, t_norm, window,
                                         autojunk=False).ratio()
                if ratio > best_ratio:
                    best_ratio = ratio
                    best_start = start
                    best_end   = end
 
        # reconstruct original-cased reference span
        matched_span = " ".join(ref_tokens[best_start:best_end])
        aligned.append(matched_span)
 
        # advance cursor (allow small overlap for repeated phrases)
        cursor = max(cursor, best_end - tolerance // 2)
 
        if (line_idx + 1) % 10 == 0:
            print(f"  aligned {line_idx + 1}/{len(trans_lines)} lines …")
 
    return aligned

In [26]:
def write_comparison(trans_lines: list[str],
                     ref_aligned: list[str],
                     path: str) -> None:
    """Write a side-by-side comparison with a simple diff indicator."""
    col = 80
    with open(path, "w", encoding="utf-8") as f:
        header = f"{'TRANSCRIPTION':<{col}}  {'REFERENCE (aligned)':<{col}}  MATCH%\n"
        f.write(header)
        f.write("─" * (col * 2 + 10) + "\n")
        for i, (t, r) in enumerate(zip(trans_lines, ref_aligned), 1):
            ratio = SequenceMatcher(None, normalize(t), normalize(r),
                                    autojunk=False).ratio()
            pct   = f"{ratio*100:5.1f}%"
            f.write(f"{t.strip():<{col}}  {r:<{col}}  {pct}\n")
    print(f"Comparison written → {path}")

In [29]:
or_trans = Path(PROJECT_ROOT, 'data/raw/AlbucE.txt')
model_transc = Path(PROJECT_ROOT, 'data/processed/transcription/ocr_kept_20260514_214446/05_garde_001_full.txt')

# Read the files first
trans_lines = or_trans.read_text(encoding="utf-8").splitlines()
ref_text = model_transc.read_text(encoding="utf-8")

ref_aligned = align_reference_to_lines(trans_lines=trans_lines, ref_text=ref_text)

  aligned 10/3522 lines …
  aligned 20/3522 lines …
  aligned 30/3522 lines …
  aligned 40/3522 lines …
  aligned 50/3522 lines …
  aligned 60/3522 lines …
  aligned 70/3522 lines …
  aligned 80/3522 lines …
  aligned 90/3522 lines …
  aligned 100/3522 lines …
  aligned 110/3522 lines …
  aligned 120/3522 lines …
  aligned 130/3522 lines …
  aligned 140/3522 lines …
  aligned 150/3522 lines …
  aligned 160/3522 lines …
  aligned 170/3522 lines …
  aligned 180/3522 lines …
  aligned 190/3522 lines …
  aligned 200/3522 lines …
  aligned 210/3522 lines …
  aligned 220/3522 lines …
  aligned 230/3522 lines …
  aligned 240/3522 lines …
  aligned 250/3522 lines …
  aligned 260/3522 lines …
  aligned 270/3522 lines …
  aligned 280/3522 lines …
  aligned 290/3522 lines …
  aligned 300/3522 lines …
  aligned 310/3522 lines …
  aligned 320/3522 lines …
  aligned 330/3522 lines …
  aligned 340/3522 lines …
  aligned 350/3522 lines …
  aligned 360/3522 lines …
  aligned 370/3522 lines …
  aligned 

In [32]:
len(ref_aligned)

3522

In [37]:
ref_aligned

['comensan las paraulas de',
 'necessari es a la fe eulu. eper las exposicios delu. eper las de claracios delu es ami uist que yeu complesca aquela a uos an a q̃st tractat lequal es partida dela operacio amn ma so es corurgia. quar la operaçio am ma des prostrada en nostre religio. e en nostie temmps de tot priuada entro que fort leu pe ric la sciencia delu. ele uistigi delu es',
 'delu. ele uistigi delu es ostat e no romasero delu sino alcimnas petitas descripcios enles libres dels aucies les q̃ls nuidero las mas. e en deuenc adaquels error. e heyssitacio entro que son clausas las entencios delu. ees elonguada la for sa delu. clart. ¶ ¶ es uist ami que yeu uiuifique aquela ain ordenacio de a quest tractat en aquela segon la nia de exposicio. ede declaracio. ede abiemacio. e perque uengua aulas',
 'abiemacio. e perque uengua aulas formas dels fer fil pus yeu le iꝰ mentz de cautheri. e dels autres instru mentz dela obra cun sia per ad dicio de la declaracio. e per preparacio delu laqua

In [52]:
import re
from difflib import SequenceMatcher

def find_boundary_in_ref(ref_tokens, ref_norm, trans_line, cursor, search_window=30):
    """Find where a transcription line starts in the reference tokens."""
    t_tokens = trans_line.strip().split()
    if not t_tokens:
        return cursor
    
    # Use first few words to anchor the start
    anchor = [t.lower().strip('.,;:') for t in t_tokens[:5]]
    anchor_len = len(anchor)
    
    best_ratio = -1
    best_pos = cursor
    end = min(len(ref_norm), cursor + search_window)
    
    for i in range(cursor, end):
        window = ref_norm[i:i + anchor_len]
        ratio = SequenceMatcher(None, anchor, window, autojunk=False).ratio()
        if ratio > best_ratio:
            best_ratio = ratio
            best_pos = i
    
    return best_pos


def align_reference_to_lines(trans_lines, ref_text):
    ref_tokens = ref_text.split()
    ref_norm = [re.sub(r'[.,;:\'\"]', '', t.lower()) for t in ref_tokens]
    
    # Find the start position in ref_tokens for each transcription line
    boundaries = []
    cursor = 0
    for i, line in enumerate(trans_lines):
        if not line.strip():
            boundaries.append(cursor)
            continue
        pos = find_boundary_in_ref(ref_tokens, ref_norm, line, cursor, search_window=30)
        boundaries.append(pos)
        cursor = pos + 1  # move forward
        print(f"  Line {i+1:3d}: matched at token {pos} → '{' '.join(ref_tokens[pos:pos+6])}…'")
    
    # Slice reference between boundaries
    aligned = []
    for i, start in enumerate(boundaries):
        end = boundaries[i + 1] if i + 1 < len(boundaries) else len(ref_tokens)
        span = " ".join(ref_tokens[start:end]).strip()
        aligned.append(span)
    
    return aligned

In [53]:
or_trans = Path(PROJECT_ROOT, 'data/raw/AlbucE.txt')
model_transc = Path(PROJECT_ROOT, 'data/processed/transcription/ocr_kept_20260514_214446/05_garde_001_full.txt')

# Read the files first
trans_lines = or_trans.read_text(encoding="utf-8").splitlines()
ref_text = model_transc.read_text(encoding="utf-8")
result = align_reference_to_lines(trans_lines=trans_lines, ref_text=ref_text)

  Line   1: matched at token 0 → 'yssi comensan las paraulas de de…'
  Line   2: matched at token 1 → 'comensan las paraulas de de albucasin…'
  Line   3: matched at token 12 → 'fill necessari es a la fe…'
  Line   4: matched at token 13 → 'necessari es a la fe eulu.…'
  Line   5: matched at token 14 → 'es a la fe eulu. eper…'
  Line   6: matched at token 15 → 'a la fe eulu. eper las…'
  Line   7: matched at token 16 → 'la fe eulu. eper las exposicios…'
  Line   8: matched at token 28 → 'es ami uist que yeu complesca…'
  Line   9: matched at token 57 → 'prostrada en nostre religio. e en…'
  Line  10: matched at token 58 → 'en nostre religio. e en nostie…'
  Line  11: matched at token 59 → 'nostre religio. e en nostie temmps…'
  Line  12: matched at token 60 → 'religio. e en nostie temmps de…'
  Line  13: matched at token 61 → 'e en nostie temmps de tot…'
  Line  14: matched at token 62 → 'en nostie temmps de tot priuada…'
  Line  15: matched at token 78 → 'uistigi delu es ostat e no…'


In [64]:
import re
from difflib import SequenceMatcher

def align_reference_to_lines(
    trans_lines,
    ref_text,
    anchor_words=5,          # how many words from the START of each trans line to use as anchor
                             # ↑ more = more precise match, but fails if first words are messy
                             # ↓ fewer = more forgiving, but more ambiguous matches
    search_window=80,        # how many ref tokens AHEAD of cursor to search for the anchor
                             # ↑ larger = handles big gaps/insertions in ref, but slower & riskier
                             # ↓ smaller = faster, but misses anchors if ref has extra words
    min_advance=3,           # minimum tokens cursor moves forward after each match
                             # ↑ larger = prevents lines re-matching the same region (fixes repeats)
                             # ↓ smaller = allows very short lines to match close together
    strip_chars='.,;:\'\"',  # punctuation removed before comparing tokens
):
    """
    Cuts the reference text at boundaries that mirror the transcription line breaks.
    All reference text is preserved — it's just sliced differently.
    """
    ref_tokens = ref_text.split()
    ref_norm = [re.sub(f'[{re.escape(strip_chars)}]', '', t.lower()) for t in ref_tokens]

    boundaries = []
    cursor = 0

    for i, line in enumerate(trans_lines):
        if not line.strip():
            boundaries.append(cursor)
            continue

        t_tokens = line.strip().split()

        # Anchor: first N words of the transcription line, normalised
        anchor = [re.sub(f'[{re.escape(strip_chars)}]', '', t.lower())
                  for t in t_tokens[:anchor_words]]
        anchor_len = len(anchor)

        best_ratio = -1
        best_pos = cursor
        end = min(len(ref_norm), cursor + search_window)

        for j in range(cursor, end):
            window = ref_norm[j:j + anchor_len]
            ratio = SequenceMatcher(None, anchor, window, autojunk=False).ratio()
            if ratio > best_ratio:
                best_ratio = ratio
                best_pos = j

        boundaries.append(best_pos)

        # Advance cursor by at least min_advance to avoid re-matching same region
        cursor = max(cursor + min_advance, best_pos + 1)

        print(f"  Line {i+1:3d} (ratio={best_ratio:.2f}): "
              f"token {best_pos} → '{' '.join(ref_tokens[best_pos:best_pos+6])}…'")

    # Slice reference between boundaries — all tokens preserved
    aligned = []
    for i, start in enumerate(boundaries):
        end = boundaries[i + 1] if i + 1 < len(boundaries) else len(ref_tokens)
        aligned.append(" ".join(ref_tokens[start:end]).strip())

    return aligned

In [78]:
or_trans = Path(PROJECT_ROOT, 'data/raw/AlbucE.txt')
model_transc = Path(PROJECT_ROOT, 'data/processed/transcription/ocr_kept_20260515_104644/05_garde_001_full.txt')
# Read the files first
trans_lines = or_trans.read_text(encoding="utf-8")
ref_text = model_transc.read_text(encoding="utf-8").splitlines()
#result = align_reference_to_lines(trans_lines=trans_lines, ref_text=ref_text)

result = align_reference_to_lines(
    trans_lines=ref_text,#trans_lines,
    ref_text=trans_lines,#ref_text,
    anchor_words=5,
    search_window=10,
    min_advance=8,   # increase this to stop lines overlapping
)

  Line   1 (ratio=0.80): token 0 → 'Ayssi comensan las paraulas de Albucasim…'
  Line   2 (ratio=0.00): token 8 → ', filhs , pus yeu he…'
  Line   3 (ratio=0.25): token 17 → 'aquest libre , le qual es…'
  Line   4 (ratio=0.00): token 24 → 'derier de la sciencia de medicina…'
  Line   5 (ratio=0.33): token 32 → 'le compliment de lu so consequit…'
  Line   6 (ratio=0.25): token 44 → 'per las exposicios de lu ,…'
  Line   7 (ratio=0.00): token 48 → 'lu , e per las declaracios…'
  Line   8 (ratio=0.00): token 56 → ', es a mi vist que…'
  Line   9 (ratio=0.33): token 69 → 'tractat , le qual es partida…'
  Line  10 (ratio=0.33): token 79 → 'ma , so es cyrurgia .…'
  Line  11 (ratio=0.25): token 83 → 'cyrurgia . Quar la operacio am…'
  Line  12 (ratio=0.00): token 88 → 'am ma es prostrada en nostre…'
  Line  13 (ratio=0.00): token 96 → 'en nostre temps de tot privada…'
  Line  14 (ratio=0.20): token 104 → 'que fort leu peric la sciencia…'
  Line  15 (ratio=0.20): token 114 → 'le vistigi de lu

In [79]:
result

['Ayssi comensan las paraulas de Albucasim . O',
 ', filhs , pus yeu he a vos complit',
 'aquest libre , le qual es le',
 'derier de la sciencia de medicina , per',
 'le compliment de lu so consequit la fi en lu , e',
 'per las exposicios de',
 'lu , e per las declaracios de lu',
 ', es a mi vist que yeu complesca aquel a vos am aquest',
 'tractat , le qual es partida de la operacio am',
 'ma , so es',
 'cyrurgia . Quar la operacio',
 'am ma es prostrada en nostre regio e',
 'en nostre temps de tot privada , entro',
 'que fort leu peric la sciencia de lu , e',
 'le vistigi de lu es ostat',
 '; e no romasero de lu sino alcunas',
 'petitas descripcios en les libres de los Antics , les quals mudero las mas e endevenc ad',
 'aquels',
 'error e heyssitacio entro que son clausas las',
 'entencios de lu e es elonguada la forsa',
 "de lu e l ' art . E es vist a mi que yeu vivifique aquela am",
 'ordenacio de aquest tractat en aquela segon',
 'la via',
 'de exposicio e de declaracio e de abrevi

In [69]:
1 +0.5+1.67+1.88+2.93+2.05

10.030000000000001

In [86]:
import re
from difflib import SequenceMatcher

def align_reference_to_ocr_lines(
    ocr_lines,
    ref_text,
    max_overlap=50,    # How many words to look back
    min_overlap=5,     # Minimum words to consider
):
    """
    Align reference text to match OCR line breaks.
    Each OCR line is matched to a segment of reference text.
    """
    # Tokenize reference
    ref_words = ref_text.split()
    ref_norm = [re.sub(r'[.,;:\'\"]', '', w.lower()) for w in ref_words]
    
    aligned_lines = []
    ref_pos = 0
    
    for i, ocr_line in enumerate(ocr_lines):
        if not ocr_line.strip():
            aligned_lines.append("")
            continue
        
        # Normalize OCR line
        ocr_words = ocr_line.strip().split()
        ocr_norm = [re.sub(r'[.,;:\'\"]', '', w.lower()) for w in ocr_words]
        ocr_len = len(ocr_words)
        
        # Search for best match starting near current position
        # Allow looking back a bit to handle split differences
        search_start = max(0, ref_pos - max_overlap)
        search_end = min(len(ref_words), ref_pos + max_overlap + ocr_len * 2)
        
        best_score = -1
        best_start = ref_pos
        best_end = ref_pos + ocr_len
        
        # Try different segment lengths and positions
        for start in range(search_start, search_end):
            for length in range(max(min_overlap, ocr_len - 3), ocr_len + 10):
                end = start + length
                if end > len(ref_words):
                    break
                
                # Compare OCR line to this reference segment
                ref_segment = ref_norm[start:end]
                score = SequenceMatcher(None, ocr_norm, ref_segment).ratio()
                
                # Penalize segments that are too far from current position
                distance_penalty = abs(start - ref_pos) * 0.01
                
                adjusted_score = score - distance_penalty
                
                if adjusted_score > best_score:
                    best_score = adjusted_score
                    best_start = start
                    best_end = end
        
        # Extract the matched reference segment
        aligned_line = " ".join(ref_words[best_start:best_end])
        aligned_lines.append(aligned_line)
        
        # Move cursor forward
        ref_pos = best_end
        
        # Debug output
        if best_score > 0.5:
            print(f"Line {i+1:3d} (score={best_score:.2f}): '{aligned_line[:50]}...'")
        else:
            print(f"Line {i+1:3d} (score={best_score:.2f}) - WEAK: '{ocr_line.strip()[:30]}...'")
    
    return aligned_lines

In [88]:
# Read files
or_trans = Path(PROJECT_ROOT, 'data/raw/AlbucE.txt')
model_transc = Path(PROJECT_ROOT, 'data/processed/transcription/ocr_kept_20260515_104644/05_garde_001_full.txt')

with open(or_trans, 'r', encoding='utf-8') as f:
    ref_text = f.read()

with open(model_transc, 'r', encoding='utf-8') as f:
    ocr_lines = f.read().splitlines()

# Align
aligned_ref = align_reference_to_ocr_lines(ocr_lines, ref_text)

# Save aligned reference
with open(Path(PROJECT_ROOT, 'tests/ocr/AlbucE_aligned.txt'), 'w', encoding='utf-8') as f:
    for line in aligned_ref:
        f.write(line + '\n')

print(f"\nAligned {len(aligned_ref)} lines")

Line   1 (score=0.80): 'Ayssi comensan las paraulas de...'
Line   2 (score=0.28) - WEAK: 'de albucasin...'
Line   3 (score=0.27) - WEAK: 'fil pus peu le...'
Line   4 (score=0.24) - WEAK: 'auos complit...'
Line   5 (score=0.71): 'aquest libre , le qual...'
Line   6 (score=0.88): 'qual es le derier de...'
Line   7 (score=0.00) - WEAK: 'rier dela scien...'
Line   8 (score=0.25) - WEAK: 'cia de nlediciñ...'
Line   9 (score=0.45) - WEAK: 'per le complimĩt...'
Line  10 (score=0.25) - WEAK: 'delu so conseqͥt...'
Line  11 (score=0.19) - WEAK: 'la fe eulu. eper...'
Line  12 (score=0.57): 'e per las exposicios de...'
Line  13 (score=0.39) - WEAK: 'delu. eper las declaracios del...'
Line  14 (score=0.51): 'vist que yeu complesca aquel a...'
Line  15 (score=0.56): 'tractat , le qual es partida de la operacio am...'
Line  16 (score=0.70): 'ma , so es cyrurgia . Quar la operacio am ma...'
Line  17 (score=0.75): 'es prostrada en nostre regio e en...'
Line  18 (score=0.69): 'de tot privada , entro q

In [90]:
import re
from difflib import SequenceMatcher

def align_reference_to_ocr_lines_fixed(
    ocr_lines,
    ref_text,
    anchor_words=5,
    search_ahead=80,
    min_advance=2,        # Ensure cursor always moves forward
):
    """
    Align reference to OCR lines WITHOUT overlaps or gaps.
    """
    ref_words = ref_text.split()
    ref_norm = [re.sub(r'[.,;:\'\"]', '', w.lower()) for w in ref_words]
    
    aligned_lines = []
    cursor = 0  # Tracks where we are in reference
    
    for i, ocr_line in enumerate(ocr_lines):
        if not ocr_line.strip():
            aligned_lines.append("")
            continue
        
        ocr_words = ocr_line.strip().split()
        ocr_norm = [re.sub(r'[.,;:\'\"]', '', w.lower()) for w in ocr_words]
        
        # Create anchor from first N words of OCR line
        anchor_len = min(anchor_words, len(ocr_norm))
        anchor = ocr_norm[:anchor_len]
        
        # Search ONLY forward from cursor (no backtracking!)
        search_start = cursor
        search_end = min(len(ref_norm), cursor + search_ahead)
        
        best_score = -1
        best_match_start = cursor
        best_match_end = cursor + anchor_len
        
        # Try to find anchor at different positions
        for pos in range(search_start, search_end - anchor_len + 1):
            candidate = ref_norm[pos:pos + anchor_len]
            score = SequenceMatcher(None, anchor, candidate).ratio()
            
            if score > best_score:
                best_score = score
                best_match_start = pos
                # Estimate how many words this OCR line needs
                best_match_end = pos + max(anchor_len, len(ocr_words) - 2)
        
        # Extract the aligned reference segment
        # Extend to find natural boundary (next punctuation or similar length to OCR)
        segment_end = best_match_end
        while segment_end < len(ref_words) and (segment_end - best_match_start) < len(ocr_words) + 5:
            if ref_words[segment_end] in ['.', ',', ';', ':']:
                segment_end += 1
                break
            segment_end += 1
        
        aligned_segment = " ".join(ref_words[best_match_start:segment_end])
        aligned_lines.append(aligned_segment)
        
        # CRITICAL: Move cursor forward, never backward
        # Must advance by at least min_advance to prevent overlaps
        cursor = max(cursor + min_advance, segment_end)
        
        # Debug output
        print(f"Line {i+1:3d} (score={best_score:.2f}, cursor={best_match_start}→{segment_end}): "
              f"'{aligned_segment[:60]}...'")
    
    return aligned_lines

In [91]:

# Read files
or_trans = Path(PROJECT_ROOT, 'data/raw/AlbucE.txt')
model_transc = Path(PROJECT_ROOT, 'data/processed/transcription/ocr_kept_20260515_104644/05_garde_001_full.txt')

with open(or_trans, 'r', encoding='utf-8') as f:
    ref_text = f.read()

with open(model_transc, 'r', encoding='utf-8') as f:
    ocr_lines = f.read().splitlines()


aligned_ref = align_reference_to_ocr_lines_fixed(
    ocr_lines,
    ref_text,
    anchor_words=5,
    search_ahead=80,
    min_advance=2,        # Ensure cursor always moves forward
)

# Save aligned reference
with open(Path(PROJECT_ROOT, 'tests/ocr/AlbucE_aligned2.txt'), 'w', encoding='utf-8') as f:
    for line in aligned_ref:
        f.write(line + '\n')

print(f"\nAligned {len(aligned_ref)} lines")

Line   1 (score=0.80, cursor=0→7): 'Ayssi comensan las paraulas de Albucasim ....'
Line   2 (score=0.50, cursor=24→31): 'derier de la sciencia de medicina ,...'
Line   3 (score=0.25, cursor=31→40): 'per le compliment de lu so consequit la fi...'
Line   4 (score=0.00, cursor=40→43): 'en lu ,...'
Line   5 (score=0.33, cursor=66→71): 'vos am aquest tractat ,...'
Line   6 (score=0.75, cursor=72→81): 'qual es partida de la operacio am ma ,...'
Line   7 (score=0.00, cursor=81→85): 'so es cyrurgia ....'
Line   8 (score=0.33, cursor=97→103): 'nostre temps de tot privada ,...'
Line   9 (score=0.33, cursor=112→120): ', e le vistigi de lu es ostat...'
Line  10 (score=0.00, cursor=120→128): '; e no romasero de lu sino alcunas...'
Line  11 (score=0.25, cursor=157→166): 'e es elonguada la forsa de lu e l...'
Line  12 (score=0.50, cursor=201→208): 'am las formas de los ferramentz de...'
Line  13 (score=0.00, cursor=208→221): 'cautheri e de los autres instrumentz de la obra cum sia per ...'
Line  14 (

In [92]:
def align_by_word_count(ocr_lines, ref_text):
    """
    Simple approach: match OCR lines to reference by word count.
    Works well when OCR has correct line breaks but different wording.
    """
    ref_words = ref_text.split()
    aligned_lines = []
    cursor = 0
    
    for i, ocr_line in enumerate(ocr_lines):
        if not ocr_line.strip():
            aligned_lines.append("")
            continue
        
        ocr_word_count = len(ocr_line.strip().split())
        
        # Take approximately the same number of words from reference
        # Adjust slightly for punctuation differences
        segment_length = max(ocr_word_count - 2, min(ocr_word_count + 3, 15))
        
        # Find natural boundary near the estimated end
        estimated_end = cursor + segment_length
        actual_end = estimated_end
        
        # Look for punctuation boundary within ±5 words
        for offset in range(-5, 6):
            check_pos = estimated_end + offset
            if 0 < check_pos < len(ref_words):
                if ref_words[check_pos] in ['.', ',', ';', ':', 'e', 'en', 'de']:
                    actual_end = check_pos + 1
                    break
        
        # Extract segment
        segment = " ".join(ref_words[cursor:actual_end])
        aligned_lines.append(segment)
        
        # Move cursor forward (no overlap!)
        cursor = actual_end
        
        print(f"Line {i+1:3d} (words: {len(segment.split())}): '{segment[:60]}...'")
    
    return aligned_lines

In [93]:
# Read files
or_trans = Path(PROJECT_ROOT, 'data/raw/AlbucE.txt')
model_transc = Path(PROJECT_ROOT, 'data/processed/transcription/ocr_kept_20260515_104644/05_garde_001_full.txt')

with open(or_trans, 'r', encoding='utf-8') as f:
    ref_text = f.read()

with open(model_transc, 'r', encoding='utf-8') as f:
    ocr_lines = f.read().splitlines()


aligned_ref = align_by_word_count(ocr_lines, ref_text)

# Save aligned reference
with open(Path(PROJECT_ROOT, 'tests/ocr/AlbucE_aligned3.txt'), 'w', encoding='utf-8') as f:
    for line in aligned_ref:
        f.write(line + '\n')

print(f"\nAligned {len(aligned_ref)} lines")

Line   1 (words: 5): 'Ayssi comensan las paraulas de...'
Line   2 (words: 2): 'Albucasim ....'
Line   3 (words: 4): 'O , filhs ,...'
Line   4 (words: 9): 'pus yeu he a vos complit aquest libre ,...'
Line   5 (words: 6): 'le qual es le derier de...'
Line   6 (words: 3): 'la sciencia de...'
Line   7 (words: 2): 'medicina ,...'
Line   8 (words: 4): 'per le compliment de...'
Line   9 (words: 6): 'lu so consequit la fi en...'
Line  10 (words: 2): 'lu ,...'
Line  11 (words: 5): 'e per las exposicios de...'
Line  12 (words: 2): 'lu ,...'
Line  13 (words: 7): 'e per las declaracios de lu ,...'
Line  14 (words: 14): 'es a mi vist que yeu complesca aquel a vos am aquest tractat...'
Line  15 (words: 10): 'le qual es partida de la operacio am ma ,...'
Line  16 (words: 12): 'so es cyrurgia . Quar la operacio am ma es prostrada en...'
Line  17 (words: 10): 'nostre regio e en nostre temps de tot privada ,...'
Line  18 (words: 8): 'entro que fort leu peric la sciencia de...'
Line  19 (words: 10): 'lu 

In [95]:
import re
from difflib import SequenceMatcher

def align_reference_to_ocr(ocr_lines, ref_text):
    """
    Splits reference text to match OCR line breaks.
    Guarantees: no missing words, no duplicates, strict left-to-right coverage.
    """
    ref_words = ref_text.split()
    ref_norm = [re.sub(r'[.,;:\'\"]', '', w.lower()) for w in ref_words]
    
    # Flatten OCR and track line boundaries
    ocr_flat = [w for line in ocr_lines for w in line.strip().split()]
    ocr_norm = [re.sub(r'[.,;:\'\"]', '', w.lower()) for w in ocr_flat]
    
    line_end_indices = []
    idx = 0
    for line in ocr_lines:
        idx += len(line.strip().split())
        line_end_indices.append(idx)
    
    # Step 1: Map each OCR word to its best match position in reference
    ocr_to_ref = {}
    ref_cursor = 0
    max_skip = 10  # How far ahead to search for a match
    
    for ocr_idx, ocr_w in enumerate(ocr_norm):
        best_score, best_pos = 0, ref_cursor
        
        # Search forward from current cursor
        for r in range(ref_cursor, min(ref_cursor + max_skip, len(ref_norm))):
            score = SequenceMatcher(None, ocr_w, ref_norm[r]).ratio()
            if score > best_score:
                best_score, best_pos = score, r
            if score > 0.85:  # Good enough, stop searching
                break
        
        ocr_to_ref[ocr_idx] = best_pos
        ref_cursor = best_pos + 1  # Always move forward
    
    # Step 2: Convert OCR line endings to reference split points
    ref_boundaries = [0]
    for end_idx in line_end_indices[:-1]:
        # Reference position right after the last word of this OCR line
        ref_pos = ocr_to_ref.get(end_idx - 1, ref_boundaries[-1]) + 1
        ref_boundaries.append(ref_pos)
    ref_boundaries.append(len(ref_words))  # End of text
    
    # Step 3: Enforce strict ordering & minimum segment size
    for i in range(1, len(ref_boundaries)):
        ref_boundaries[i] = max(ref_boundaries[i], ref_boundaries[i-1] + 1)
    
    # Step 4: Slice reference
    aligned = []
    for i in range(len(ref_boundaries) - 1):
        segment = ref_words[ref_boundaries[i]:ref_boundaries[i+1]]
        aligned.append(" ".join(segment).strip())
    
    # Verification
    total_ref = len(ref_words)
    total_aligned = sum(len(line.split()) for line in aligned)
    if total_ref != total_aligned:
        print(f"⚠️ Coverage mismatch: Reference={total_ref}, Aligned={total_aligned}")
    
    return aligned

In [97]:

or_trans = Path(PROJECT_ROOT, 'data/raw/AlbucE.txt')
model_transc = Path(PROJECT_ROOT, 'data/processed/transcription/ocr_kept_20260515_104644/05_garde_001_full.txt')

with open(or_trans, 'r', encoding='utf-8') as f:
    ref = f.read()
with open(model_transc, 'r', encoding='utf-8') as f:
    ocr = f.read().splitlines()

# Align
aligned = align_reference_to_ocr(ocr, ref)

# Verify coverage
ref_words = ref.split()
aligned_words = [w for line in aligned for w in line.split()]
print(f"Reference: {len(ref_words)} words")
print(f"Aligned:   {len(aligned_words)} words")
print(f"Match:     {len(ref_words) == len(aligned_words)}")

# Save aligned reference
with open( Path(PROJECT_ROOT, 'tests/ocr/AlbucE_aligned.txt'), 'w', encoding='utf-8') as f:
    f.write('\n'.join(aligned))

Reference: 112537 words
Aligned:   112537 words
Match:     True


In [98]:
def align_reference_to_ocr_lines(ocr_lines, ref_text):
    """
    Splits reference text to match OCR line-by-line structure.
    Uses exact word counts → guarantees no missing/duplicated words.
    """
    ref_words = ref_text.split()
    ocr_word_counts = [len(line.strip().split()) for line in ocr_lines]
    
    aligned = []
    cursor = 0
    
    for i, count in enumerate(ocr_word_counts):
        # Safety: if reference runs out, pad with empty string
        if cursor >= len(ref_words):
            aligned.append("")
            continue
            
        take = min(count, len(ref_words) - cursor)
        aligned.append(" ".join(ref_words[cursor:cursor + take]))
        cursor += take
        
    # Attach any leftover reference words to the last line
    if cursor < len(ref_words) and aligned:
        aligned[-1] += " " + " ".join(ref_words[cursor:])
        
    return aligned

In [99]:
or_trans = Path(PROJECT_ROOT, 'data/raw/AlbucE.txt')
model_transc = Path(PROJECT_ROOT, 'data/processed/transcription/ocr_kept_20260515_104644/05_garde_001_full.txt')

with open(or_trans, 'r', encoding='utf-8') as f:
    ref = f.read()
with open(model_transc, 'r', encoding='utf-8') as f:
    ocr = f.read().splitlines()

# Align
aligned_ref = align_reference_to_ocr_lines(ocr_lines, ref)

# Verify exact line count match
print(f"OCR lines: {len(ocr_lines)}")
print(f"Aligned lines: {len(aligned_ref)}")
print(f"Match: {len(ocr_lines) == len(aligned_ref)}")

# Save for side-by-side comparison
with open(Path(PROJECT_ROOT, 'tests/ocr/AlbucE_aligned4.txt'), 'w', encoding='utf-8') as f:
    f.write('\n'.join(aligned_ref))

OCR lines: 100
Aligned lines: 100
Match: True
